# Creating Dataset Splits

For machine learning applications, datasets are typically split into training, validation, and test sets. These splits are crucial for evaluating model generalization and preventing overfitting. However, creating consistent and reproducible splits can be challenging when done ad-hoc.

This notebook demonstrates how to create deterministic dataset splits using ProteinGym's splitting functionality. We'll cover:

1. **Random splits** - for general train/validation/test divisions
2. **K-fold splits** - for cross-validation workflows
3. **Archiving splits** - for sharing and reproducibility

## Why Deterministic Splits?

Even "random" splits should be deterministic and shareable. This ensures:
- **Reproducibility**: Same results across different runs
- **Fair comparison**: All models evaluated on identical data splits
- **Collaboration**: Teams can work with the same train/test divisions

## Loading the Dataset

Let's start by loading our example dataset:

In [ ]:
from pathlib import Path
from proteingym.base import Dataset, Manifest
from proteingym.base.splits import RandomSplitter, KFoldSplitter

# Load the NEIME 2019 dataset
manifest_path = Path("../example_data/neime_2019.toml")
manifest = Manifest.from_path(manifest_path)
dataset = Dataset.from_manifest(manifest)

print(f"Loaded dataset: {dataset.name}")
print(f"Number of assay records: {len(dataset.assays[0].records)}")

## Random Splits

Random splits divide your data into training, validation, and test sets with specified proportions. The `RandomSplitter` creates deterministic splits using a fixed random seed.

### Three-way Split (Train/Validation/Test)

In [ ]:
# Create a 80/10/10 train/validation/test split
random_splitter = RandomSplitter(dataset, fractions=[0.8, 0.1, 0.1])

print(f"Split proportions: {random_splitter.fractions}")

In [ ]:
# Generate the split
split_superset = random_splitter.split()

print(f"Created {len(split_superset)} splits:")
split_names = ["Train", "Validation", "Test"]

for i, split_dataset in enumerate(split_superset):
    count = len(split_dataset.assays[0].records)
    total = len(dataset.assays[0].records)
    percentage = (count / total) * 100
    print(f"  {split_names[i]}: {count} samples ({percentage:.1f}%)")

### Two-way Split (Train/Test)

In [ ]:
# Create a simple 80/20 train/test split
simple_splitter = RandomSplitter(dataset, fractions=[0.8, 0.2])
simple_split = simple_splitter.split()

# Unpack splits directly
train_dataset, test_dataset = tuple(simple_split)

print(f"Two-way split created:")
print(f"  Train: {len(train_dataset.assays[0].records)} samples")
print(f"  Test: {len(test_dataset.assays[0].records)} samples")

## K-Fold Cross-Validation Splits

K-fold splits divide data into k equal-sized folds for cross-validation. Each fold can serve as a test set while the remaining folds form the training set.

In [ ]:
# Create 5-fold cross-validation splits
kfold_splitter = KFoldSplitter(dataset, n_splits=5, random_state=42, shuffle=True)

print(f"K-fold configuration:")
print(f"  Number of folds: {kfold_splitter.n_splits}")
print(f"  Shuffle data: {kfold_splitter.shuffle}")
print(f"  Random seed: {kfold_splitter.random_state}")

In [ ]:
# Generate the k-fold splits
kfold_superset = kfold_splitter.split()

print(f"\nCreated {len(kfold_superset)} folds:")
total_samples = len(dataset.assays[0].records)

for i, fold_dataset in enumerate(kfold_superset):
    test_count = len(fold_dataset.assays[0].records)
    train_count = total_samples - test_count
    print(f"  Fold {i+1}: {train_count} train, {test_count} test samples")

## Working with Split Datasets

Each split returns a complete Dataset object with filtered data:

In [ ]:
# Unpack the three-way split
train_dataset, val_dataset, test_dataset = tuple(split_superset)

print(f"Training dataset: {len(train_dataset.assays[0].records)} samples")
print(f"Validation dataset: {len(val_dataset.assays[0].records)} samples")
print(f"Test dataset: {len(test_dataset.assays[0].records)} samples")

# Each split is a complete Dataset object
print(f"\nTraining dataset type: {type(train_dataset)}")
print(f"Training dataset name: {train_dataset.name}")

## Archiving Splits for Reproducibility

You can save splits to share with collaborators or ensure reproducibility across experiments:

In [ ]:
# Archive the random split
archive_path = split_superset.dump(path=Path("../example_data/"))
print(f"Split archived to: {archive_path}")
print(f"Archive size: {archive_path.stat().st_size / 1024:.1f} KB")

### Loading Archived Splits

In [ ]:
# Load splits from archive
from proteingym.base.splits import DatasetSplitSuperset

loaded_splits = DatasetSplitSuperset.from_path(archive_path)

print(f"Loaded splits from archive")
print(f"Number of splits: {len(loaded_splits)}")
print(f"Original dataset name: {loaded_splits.name}")

# Verify the splits are identical
original_train = tuple(split_superset)[0]
loaded_train = tuple(loaded_splits)[0]

original_count = len(original_train.assays[0].records)
loaded_count = len(loaded_train.assays[0].records)
print(f"\nSplits are identical: {original_count == loaded_count}")

## Practical Usage Examples

### Example 1: Using Splits in ML Workflows

In [ ]:
# Example: Extract training and test data for ML
train_dataset, val_dataset, test_dataset = tuple(split_superset)

# Extract sequences and targets from training set
train_records = train_dataset.assays[0].records
train_sequences = [record[0] for record in train_records]
train_targets = [record[1] for record in train_records]

print(f"Training set: {len(train_sequences)} sequences")
print(f"First training sequence: {train_sequences[0][:20]}...")
print(f"First training target: {train_targets[0]}")

# Extract test data
test_records = test_dataset.assays[0].records
test_sequences = [record[0] for record in test_records]
test_targets = [record[1] for record in test_records]

print(f"\nTest set: {len(test_sequences)} sequences")

### Example 2: Cross-Validation Loop

In [ ]:
# Example: Cross-validation workflow
cv_scores = []

for fold_idx, test_dataset in enumerate(kfold_superset):
    # Get test data
    test_records = test_dataset.assays[0].records
    n_test = len(test_records)
    
    # Calculate training size (total - test)
    total_samples = len(dataset.assays[0].records)
    n_train = total_samples - n_test
    
    # Simulate model training and evaluation
    # (In practice, you'd train your model here)
    simulated_score = 0.85 + (fold_idx * 0.02)  # Fake score for demo
    cv_scores.append(simulated_score)
    
    print(f"Fold {fold_idx + 1}: {n_train} train, {n_test} test → Score: {simulated_score:.3f}")

mean_score = sum(cv_scores) / len(cv_scores)
print(f"\nMean CV Score: {mean_score:.3f}")

## Best Practices for Dataset Splitting

### 1. **Reproducibility**
- Always use fixed random seeds
- Archive and share your splits
- Document your splitting strategy

### 2. **Split Proportions**
- **Small datasets**: Use cross-validation instead of fixed splits
- **Medium datasets**: 70/15/15 or 80/10/10 train/val/test
- **Large datasets**: 80/20 train/test may be sufficient

### 3. **Biological Considerations**
- Consider protein families or structural similarity when splitting
- Avoid data leakage between splits
- Account for experimental batch effects

### 4. **Validation Strategy**
- Use validation sets for hyperparameter tuning
- Reserve test sets for final evaluation only
- Consider nested cross-validation for robust evaluation

## Summary

In this notebook, we've learned how to:

1. **Create random splits** with specified proportions
2. **Generate k-fold splits** for cross-validation
3. **Archive and load splits** for reproducibility
4. **Extract data** from splits for ML workflows
5. **Apply best practices** for dataset splitting

The ProteinGym splitting functionality ensures your ML experiments are reproducible and comparable across different studies.

## Next Steps

Now you can:
- Apply these splitting strategies to your own datasets
- Integrate splits into your ML training pipelines
- Share standardized splits with collaborators
- Explore advanced splitting strategies for specific biological contexts